<a href="https://colab.research.google.com/github/ishashahul/CertChain/blob/main/LSTM_Word_Generation_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Word Generation with LSTM

Word generation is a task where a model predicts the next word in a sequence based on the preceding words. Long Short-Term Memory (LSTM) networks, a type of Recurrent Neural Network (RNN), are well-suited for this task due to their ability to learn long-term dependencies in sequential data.

The pipeline typically involves the following steps:

1.  **Data Preparation**: Collecting and cleaning text data.
2.  **Tokenization**: Converting text into numerical sequences.
3.  **Model Architecture**: Defining the LSTM model structure.
4.  **Training**: Training the model on the tokenized data.
5.  **Inference**: Using the trained model to generate new words based on a seed text.

In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
import numpy as np

### 1. Data Preparation

For this demonstration, we'll use a small, predefined corpus of text. In a real-world scenario, you would use a much larger dataset, such as books, articles, or web data. The goal is to prepare the text into sequences that the model can learn from.

In [2]:
text = "Natural Language Processing is a field of artificial intelligence that focuses on the interaction between computers and human language. It involves various techniques like tokenization, parsing, and semantic analysis to enable computers to understand, interpret, and generate human language. LSTM models are powerful for sequence tasks in NLP."

# Create sequences of words from the text
corpus = text.lower().split('. ')

### 2. Tokenization

Tokenization is the process of breaking down text into smaller units (tokens), which can be words or subwords. These tokens are then mapped to numerical IDs. We also create input sequences where each sequence consists of `n-1` words as input and the `n`-th word as the target for prediction.

In [3]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
total_words = len(tokenizer.word_index) + 1

input_sequences = []
for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Pad sequences to ensure uniform length
max_sequence_len = max([len(x) for x in input_sequences])
padded_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# Create predictors (X) and labels (y)
X, y = padded_sequences[:, :-1], padded_sequences[:, -1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

### 3. Model Architecture

The model consists of an Embedding layer, an LSTM layer, and a Dense (output) layer.

*   **Embedding Layer**: Converts input word indices into dense vectors of fixed size, capturing semantic relationships.
*   **LSTM Layer**: Processes the sequences and learns long-term dependencies.
*   **Dense Layer**: The output layer with a 'softmax' activation function, which outputs a probability distribution over the entire vocabulary, indicating the likelihood of each word being the next word in the sequence.

In [6]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
import tensorflow as tf

text = "Natural Language Processing is a field of artificial intelligence that focuses on the interaction between computers and human language. It involves various techniques like tokenization, parsing, and semantic analysis to enable computers to understand, interpret, and generate human language. LSTM models are powerful for sequence tasks in NLP."

# Create sequences of words from the text
corpus = text.lower().split('. ')

tokenizer = Tokenizer()
tokenizer.fit_on_texts(corpus)
total_words = len(tokenizer.word_index) + 1

input_sequences = []
for line in corpus:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Pad sequences to ensure uniform length
max_sequence_len = max([len(x) for x in input_sequences])
padded_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# Create predictors (X) and labels (y)
X, y = padded_sequences[:, :-1], padded_sequences[:, -1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

embedding_dim = 100
model = Sequential([
    Embedding(total_words, embedding_dim),
    LSTM(150),
    Dense(total_words, activation='softmax')
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

### 4. Training

The model is compiled with an optimizer (e.g., 'adam') and a loss function suitable for multi-class classification ('categorical_crossentropy'). It is then trained on the input sequences (X) and their corresponding target words (y).

In [5]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

history = model.fit(X, y, epochs=500, verbose=0) # Set verbose to 0 to suppress output, or 1 to see progress
print(f"Training accuracy: {history.history['accuracy'][-1]:.4f}")

Training accuracy: 1.0000


### 5. Inference (Word Generation)

To generate new words, we provide a 'seed text'. The model predicts the next word, which is then appended to the seed text, and the process is repeated. This allows the model to generate a sequence of words.

In [6]:
seed_text = "Natural Language Processing"
next_words = 5

for _ in range(next_words):
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
    predicted_probs = model.predict(token_list, verbose=0)[0]
    predicted_word_index = np.argmax(predicted_probs)

    output_word = ""
    for word, index in tokenizer.word_index.items():
        if index == predicted_word_index:
            output_word = word
            break
    seed_text += " " + output_word

print(f"Generated text: {seed_text}")

Generated text: Natural Language Processing is a field of artificial
